# Working with Census Geographies

morpc-census provides tools to fetch Census geometries and data for specific places at specific resolutions. Every query is defined by two things:

- **Scope** — *where* (Franklin County, the 15-county MORPC region, the state of Ohio)
- **SumLevel** — *at what resolution* (county, census tract, place)

The results come back with a **GEOIDFQ** — a fully-qualified geographic identifier — for each geography returned.

In [ ]:
from morpc_census import (
    SCOPES,
    Scope, SumLevel, GeoIDFQ,
    geoinfo_from_scope_sumlevel, geoids_from_scope,
    fetch_geos_from_scope_sumlevel,
    resource_from_scope_sumlevel, resource_from_geometry_sumlevel,
)

## 1. Scopes — choosing where

A scope names a geographic extent. morpc-census ships with built-in scopes for the US, all 50 states, all Ohio counties, and several MORPC regions.

In [ ]:
# See all available scope names
list(SCOPES.keys())

Select scopes from the predetermined list

In [ ]:
# Look up a specific scope
SCOPES["franklin"]

In [ ]:
# The 15-county MORPC region covers multiple counties — for_param holds their FIPS codes
SCOPES["region15"]

In [ ]:
region15 = Scope('region15')

In [ ]:
region15.params

In [ ]:
region15.sql

## 2. Summary levels — choosing resolution

A `SumLevel` pairs a Census query name (e.g. `"county"`) with its three-digit summary level code (e.g. `"050"`). Construct one from either form — the other field is filled in automatically.

Optional metadata fields (`singular`, `plural`, `hierarchy_string`, `tigerweb_name`) default to `None` and can be supplied when the full description is needed.

In [ ]:
# Construct from a query name — three-digit code is filled in automatically
SumLevel("county")

In [ ]:
# Construct from a three-digit code — query name is filled in automatically; same result
SumLevel("050")

In [ ]:
# Optional metadata fields default to None when constructing by name or code alone
sl = SumLevel("county")
print(sl.name, sl.sumlevel, sl.singular, sl.plural, sl.hierarchy_string, sl.tigerweb_name, sl.current_variant)

In [ ]:
sl

In [ ]:
# Supply metadata explicitly when the full description is needed
SumLevel(
    name="county", sumlevel="050",
    singular="county", plural="counties",
    hierarchy_string="COUNTY", tigerweb_name="counties",
    current_variant='00'
)

In [ ]:
# Unrecognized names or codes raise a ValueError listing available options
try:
    SumLevel("neighborhood")
except ValueError as e:
    print(e)

## 3. Querying geographic IDs

`geoinfo_from_scope_sumlevel` queries the Census geographic info API and returns GEOIDFQs for every geography in a scope at a given resolution — without downloading geometry data. Use it when you want to enumerate geographies, check what's in a scope, or collect IDs to pass to other Census queries.

It accepts either string names or class instances for both `scope` and `sumlevel`.

> **Network required** — cells below make live calls to the Census API.

In [ ]:
# Get GEOIDFQs for all counties in the 15-county MORPC region
# When sumlevel is omitted, the scope's native level is used
geoinfo_from_scope_sumlevel("region15")

In [ ]:
# Get the same result as a DataFrame (GEO_ID and NAME columns)
geoinfo_from_scope_sumlevel("region15", output="table")

In [ ]:
# Drill down to a finer resolution: census tracts within Franklin County
geoinfo_from_scope_sumlevel("franklin", "tract", output="table")

In [ ]:
# Scope and SumLevel objects can be passed directly
franklin = Scope("franklin")
tract = SumLevel("tract")
geoinfo_from_scope_sumlevel(franklin, tract, output="table")

### `geoids_from_scope`

`geoids_from_scope` is a lower-level helper that returns GEOIDFQs for the geographies that *define* a scope — the actual counties, CBSAs, or other units that the scope is built from. Unlike `geoinfo_from_scope_sumlevel`, it does not accept a `sumlevel` argument.

In [ ]:
# Get the defining GEOIDFQs for the franklin scope (the county itself)
geoids_from_scope("franklin")

In [ ]:
# The region15 scope is defined by 15 county GEOIDFQs
geoids_from_scope("region15")

In [ ]:
# Each GEOIDFQ can be parsed to inspect its components
GeoIDFQ.parse(geoids_from_scope("franklin")[0])

## 4. Fetching geometries

`fetch_geos_from_scope_sumlevel(scope, sumlevel)` is the primary function for downloading spatial data. Pass a scope name and an optional sumlevel name — it returns a GeoDataFrame.

> **Network required** — the cells below make live calls to the Census API and TIGERweb REST API.

In [ ]:
# Get county boundaries for the 15-county MORPC region
# Omitting scale returns geographies at the scope's own level (counties)
region_counties = fetch_geos_from_scope_sumlevel(scope="region15")
region_counties

In [ ]:
region_counties.plot()

In [ ]:
# Get all census tracts within Franklin County
franklin_tracts = fetch_geos_from_scope_sumlevel(scope=franklin, sumlevel=tract)
franklin_tracts.plot()

## 5. TIGERweb resources

`resource_from_scope_sumlevel` and `resource_from_geometry_sumlevel` build configured TIGERweb REST API query objects without fetching data immediately. Use them when you need to inspect or archive the query before running it, or when you want to pass the resource to `morpc.rest_api.gdf_from_resource` directly.

> **Network required** — fetching from the resource makes a live call to the TIGERweb REST API.

In [ ]:
# Build a resource for census tracts within Franklin County
# The resource captures the URL, WHERE clause, and outfields — nothing is fetched yet
tract_resource = resource_from_scope_sumlevel("franklin", "tract")
tract_resource

In [ ]:
# Inspect the components of the resource
meta = tract_resource.to_dict()['_metadata']
print("path:      ", tract_resource.path)
print("where:     ", meta['params']['where'])
print("outfields: ", meta['params']['outFields'])

In [ ]:
# Fetch geometries from the resource — returns a GeoDataFrame
import morpc.rest_api as rest_api

franklin_tracts_tw = rest_api.gdf_from_resource(tract_resource)
franklin_tracts_tw.head()

In [ ]:
# Build a resource filtered to the bounding box of an existing GeoDataFrame
# Useful when you have a custom boundary and want all intersecting geographies
# (uses region_counties fetched in section 4)
region_tract_resource = resource_from_geometry_sumlevel(
    region_counties, "region15", "tract"
)
meta = region_tract_resource.to_dict()['_metadata']
print("path:      ", region_tract_resource.path)
print("geometry:  ", meta['params']['geometry'][:60], "...")

## 6. Working with GEOIDFQs

Every geography returned by the Census API has a **GEOIDFQ** (fully-qualified geographic identifier) — a string that encodes the geography type and its component codes.

For example, `1400000US39041010100` is the GEOIDFQ for census tract 101 in Delaware County, Ohio:

```
140  00  00  US  39    041     010100
─┬─  ─┬  ─┬  ─┬  ─┬─  ─┬─    ──┬──
 │    │   │   │   │    │       tract
 │    │   │   │   │    county
 │    │   │   │   state
 │    │   │   literal
 │    │   geocomp
 │    variant
 sumlevel (140 = census tract)
```

`GeoIDFQ` lets you parse, inspect, and construct these strings.

In [ ]:
# Parse a GEOIDFQ returned by the Census API
tract = GeoIDFQ.parse("1400000US39041010100")
tract

In [ ]:
tract.sumlevel.name

In [ ]:
tract.sumlevel.hierarchy_string

In [ ]:
# Geographic components are accessible as direct attributes
print(tract.state, tract.county, tract.tract)

# .parts returns the same data as a dict (backward-compatible)
tract.parts

In [ ]:
# .geoid is the short-form ID used in REST API queries (the part after 'US')
tract.geoid

In [ ]:
# Build a GEOIDFQ from component codes using keyword arguments
county = GeoIDFQ.build("050", state="39", county="049")
str(county)    # Franklin County, Ohio

In [ ]:
# For geographies with variant codes, pass variant explicitly
# Variant = Congress number minus 100, so the 119th Congress = "19"
cd = GeoIDFQ.build("500", variant="19", state="39", cd="12")
str(cd)    # Ohio's 12th Congressional District, 119th Congress

In [ ]:
# GEOIDFQs from a fetch can be parsed directly from the GEO_ID column
# (network required — uses region_counties from section 4)
geoidfqs = region_counties["GEOIDFQ"].tolist()
parsed = [GeoIDFQ.parse(g) for g in geoidfqs]

# Access geographic components as direct attributes
[(g.state, g.county) for g in parsed]